In [1]:

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ParameterGrid
import statsmodels.api as sm


In [4]:
data_folder = Path.cwd().parent / "data" / "processed"

df = pd.read_csv(
    data_folder / "weather_dataset_2024_2026.csv", 
    parse_dates=["time"]
)

In [5]:

df["hour_of_day"] = df["time"].dt.hour
df["hour_sin"] = np.sin(2 * np.pi * df["hour_of_day"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour_of_day"] / 24)

df = df.sort_values("time").set_index("time")
df = df.reset_index()

train = df[df["time"].dt.year == 2024]
val = df[df["time"].dt.year == 2025]
test = df[df["time"].dt.year == 2026]

features = [
    "temperature_ECMWF",
    "temperature_GFS",
    "temperature_ICON_EU",
    "hour_sin",
    "hour_cos",
]

target = "actual_temperature"

X_train = train[features]
y_train = train[target]

X_val = val[features]
y_val = val[target]

In [17]:
def model_mae_and_rmse(pred, y, model_name):
    print(f"\n{model_name}")
    print("MAE:", mean_absolute_error(y, pred))
    print("RMSE:", root_mean_squared_error(y, pred))

In [9]:

X = sm.add_constant(train[features])
y = train[target]

ols = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 24})
print(ols.summary())

X_val_c = sm.add_constant(X_val, has_constant="add")
ols_pred = ols.predict(X_val_c)

                            OLS Regression Results                            
Dep. Variable:     actual_temperature   R-squared:                       0.919
Model:                            OLS   Adj. R-squared:                  0.919
Method:                 Least Squares   F-statistic:                     1732.
Date:                Wed, 25 Nov 2026   Prob (F-statistic):               0.00
Time:                        11:26:18   Log-Likelihood:                -3489.4
No. Observations:                2208   AIC:                             6991.
Df Residuals:                    2202   BIC:                             7025.
Df Model:                           5                                         
Covariance Type:                  HAC                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                   0.0141    

In [18]:
lr = LinearRegression()

lr.fit(X_train, y_train)

lr_pred = lr.predict(X_val)

model_mae_and_rmse(lr_pred, y_val, "Linear regression")
model_mae_and_rmse(ols_pred, y_val, "Linear regression wiht HAC errors")



Linear regression
MAE: 0.9265575786339546
RMSE: 1.199126026952906

Linear regression wiht HAC errors
MAE: 0.9265575786339805
RMSE: 1.1991260269529251


In [ ]:
comparison = pd.DataFrame({
    "Actual": y_val.values,
    "Predicted": lr_pred,
    "ICON-EU": val["temperature_ICON_EU"].values
})

display(comparison.head(10))

In [ ]:
daily = pd.DataFrame({
    "time": val["time"].values,
    "Actual": y_val.values,
    "Linear Regression": val_pred,
    "ICON-EU": val["temperature_ICON_EU"].values
})

daily = daily.set_index("time").resample("D").mean()

plt.figure(figsize=(12, 5))

plt.plot(daily.index, daily["Actual"], label="Actual", color="black")
plt.plot(daily.index, daily["Linear Regression"], label="Linear Regression", color="orange")
plt.plot(daily.index, daily["ICON-EU"], label="ICON-EU", color="steelblue", alpha=0.7)

plt.title("Daily Average Temperature – Summer 2025")
plt.xlabel("Date")
plt.ylabel("Temperature (°C)")
plt.legend()

plt.tight_layout()
plt.show()

In [20]:
model_mae_and_rmse(val["temperature_ICON_EU"], y_val, "ICON-EU")
model_mae_and_rmse(lr_pred, y_val, "Linear regression")


ICON-EU
MAE: 1.0879528985507245
RMSE: 1.4104103113551076

Linear regression
MAE: 0.9265575786339546
RMSE: 1.199126026952906


In [ ]:
for feature, coefficient in zip(features, lr.coef_):
    print(f"{feature}: {coefficient:.4f}")

print(f"Intercept: {lr.intercept_:.4f}")

In [21]:
rf = RandomForestRegressor(
    n_estimators=250,
    random_state=42,
    max_depth=5,
    min_samples_leaf=10,
    max_features=1.0
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_val)

model_mae_and_rmse(rf_pred, y_val, "Random forest")


Random forest
MAE: 1.0112299927710833
RMSE: 1.3192302397184288
